# Train v22 on Workstation

This notebook extracts the packaged code locally and creates terminal commands for v22. Run the generated PowerShell scripts for long jobs so progress and failures are visible directly in the terminal. Keep API keys in environment variables or a local `.env`; do not paste secrets into cells.

In [ ]:
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

VERSION = 'v22'
DEFAULT_DRIVE_CODE_DIR = Path('H:/My Drive/quant-research-workbench/workstation_code/v22')
DRIVE_CODE_DIR = Path(os.environ.get('QW_DRIVE_CODE_DIR', str(DEFAULT_DRIVE_CODE_DIR)))
PACKAGE_ZIP = DRIVE_CODE_DIR / f'inhouse_transformer_{VERSION}_workstation.zip'
MANIFEST_PATH = DRIVE_CODE_DIR / 'workstation_manifest.json'
LOCAL_CODE_ROOT = Path(f'D:/TradingCodes/quant-research-workbench-{VERSION}-runtime')

assert PACKAGE_ZIP.exists(), f'Missing package: {PACKAGE_ZIP}'
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text())
print(json.dumps(manifest, indent=2))

if LOCAL_CODE_ROOT.exists():
    shutil.rmtree(LOCAL_CODE_ROOT)
LOCAL_CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as package:
    package.extractall(LOCAL_CODE_ROOT)
sys.path.insert(0, str(LOCAL_CODE_ROOT))
print('installed code at', LOCAL_CODE_ROOT)


In [ ]:
# Edit FLATFILES_ROOT if you copy data from the HDD/Drive path to local SSD/NVMe.
FLATFILES_ROOT = Path(manifest['default_flatfiles_root'])
CACHE_ROOT = Path(manifest['default_cache_root'])
OUTPUT_ROOT = Path(manifest['default_output_root'])
print('flatfiles root:', FLATFILES_ROOT, 'exists=', FLATFILES_ROOT.exists())
print('cache root:', CACHE_ROOT)
print('output root:', OUTPUT_ROOT)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess
import time

RUN_LOG_DIR = OUTPUT_ROOT / 'workstation_logs'
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)

def ps_quote(value):
    text = str(value).replace('`', '``').replace('"', '`"')
    return f'"{text}"'

def terminal_command(script_path, args):
    return ' '.join([ps_quote(sys.executable), '-u', ps_quote(script_path), *[ps_quote(arg) for arg in args]])

def write_command_script(label, script_path, args):
    script_path = Path(script_path)
    command = terminal_command(script_path, args)
    ps1_path = LOCAL_CODE_ROOT / f'run_{label}.ps1'
    log_path = RUN_LOG_DIR / f'{VERSION}_{label}.log'
    py_path = str(LOCAL_CODE_ROOT).replace("'", "''")
    ps1 = (
        "$ErrorActionPreference = 'Stop'\n"
        "$env:PYTHONUNBUFFERED = '1'\n"
        f"$env:PYTHONPATH = '{py_path}' + [System.IO.Path]::PathSeparator + $env:PYTHONPATH\n"
        f"{command} 2>&1 | Tee-Object -FilePath {ps_quote(log_path)}\n"
        "if ($LASTEXITCODE -ne 0) { throw \"Command failed with exit code $LASTEXITCODE\" }\n"
    )
    ps1_path.write_text(ps1, encoding='utf-8')
    print('=' * 96)
    print(f'{label.upper()} command generated')
    print('PowerShell script:', ps1_path)
    print('Log file:', log_path)
    print('Run this in PowerShell:')
    print('& ' + ps_quote(ps1_path))
    print('Direct command equivalent:')
    print(command)
    print('=' * 96)
    return ps1_path

def run_script_streamed(label, script_path, args):
    script_path = Path(script_path)
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    log_path = RUN_LOG_DIR / f'{VERSION}_{label}_{timestamp}.log'
    command = [sys.executable, '-u', str(script_path), *map(str, args)]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONPATH'] = str(LOCAL_CODE_ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    print('=' * 96, flush=True)
    print(f'*** {label.upper()} START | {time.strftime("%Y-%m-%d %H:%M:%S")}', flush=True)
    print('cwd:', LOCAL_CODE_ROOT, flush=True)
    print('script:', script_path, flush=True)
    print('log:', log_path, flush=True)
    print('command:', ' '.join(command), flush=True)
    print('=' * 96, flush=True)
    start = time.perf_counter()
    tail = []
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(LOCAL_CODE_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
            tail.append(line.rstrip())
            if len(tail) > 40:
                tail.pop(0)
        return_code = process.wait()
    elapsed = time.perf_counter() - start
    print('=' * 96, flush=True)
    print(f'*** {label.upper()} END | return_code={return_code} | elapsed_sec={elapsed:.1f} | log={log_path}', flush=True)
    print('=' * 96, flush=True)
    if return_code != 0:
        print('Last output lines before failure:', flush=True)
        for line in tail[-20:]:
            print(line, flush=True)
        raise RuntimeError(f'{label} failed with return code {return_code}. Full log: {log_path}')
    return log_path

print('command helpers ready; generated scripts/logs will be written under', LOCAL_CODE_ROOT, 'and', RUN_LOG_DIR)


In [ ]:
%pip install polars pyarrow wandb torchinfo torchview graphviz


Optional: run the profiling cell before choosing chunk/cap settings. This is available for v22 and skipped for older versions.

In [ ]:
RUN_PROFILE_IN_NOTEBOOK = False  # Prefer running the printed PowerShell script in a terminal.
PROFILE_PROCESSES = int(manifest.get('default_preprocess_processes', 8))
POLARS_THREADS_PER_PROCESS = int(manifest.get('default_polars_threads_per_process', 2))
profile_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'profile_event_chunks.py'
profile_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['validation_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', '100,250,500,1000',
    '--caps', '64,128,256,512',
    '--processes', str(PROFILE_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]
write_command_script('profile', profile_py, profile_args)
if RUN_PROFILE_IN_NOTEBOOK:
    run_script_streamed('profile', profile_py, profile_args)
else:
    print('Profiling was not started in the notebook. Run the printed PowerShell command in a terminal.')


Optional but recommended: prebuild the microstructure Parquet cache before training. This is the slow CSV decompression step; later training epochs reuse the cache.

In [ ]:
RUN_PREPROCESS_IN_NOTEBOOK = False  # Prefer running the printed PowerShell script in a terminal.
PREPROCESS_PROCESSES = int(manifest.get('default_preprocess_processes', 8))
POLARS_THREADS_PER_PROCESS = int(manifest.get('default_polars_threads_per_process', 2))
REBUILD_PREPROCESS_CACHE = False

preprocess_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'preprocess_event_chunks.py'
preprocess_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['test_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', '250',
    '--max-quote-events', '96',
    '--max-trade-events', '64',
    '--max-total-events', '128',
    '--processes', str(PREPROCESS_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]
if REBUILD_PREPROCESS_CACHE:
    preprocess_args.append('--rebuild-cache')

write_command_script('preprocess', preprocess_py, preprocess_args)
if RUN_PREPROCESS_IN_NOTEBOOK:
    run_script_streamed('preprocess', preprocess_py, preprocess_args)
else:
    print('Preprocess was not started in the notebook. Run the printed PowerShell command in a terminal.')


In [ ]:
RUN_TRAINING_IN_NOTEBOOK = False  # Prefer running the printed PowerShell script in a terminal.
BATCH_SIZE = int(manifest.get('default_batch_size', 4096))
EPOCHS = int(manifest.get('default_epochs', 3))
NUM_WORKERS = int(manifest.get('default_num_workers', 8))
PREFETCH_FACTOR = int(manifest.get('default_prefetch_factor', 4))
MAX_STEPS = 0
COUNT_COVERAGE = False
DRY_RUN = False
REBUILD_CACHE = False

train_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'train.py'
args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--train-start-date', manifest['train_start_date'],
    '--train-end-date', manifest['train_end_date'],
    '--validation-start-date', manifest['validation_start_date'],
    '--validation-end-date', manifest['validation_end_date'],
    '--test-start-date', manifest['test_start_date'],
    '--test-end-date', manifest['test_end_date'],
    '--device', 'cuda',
    '--output-root', str(OUTPUT_ROOT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--tickers', manifest.get('tickers', 'ALL'),
    '--checkpoint-policy', 'last_only',
    '--wandb-entity', manifest['wandb_entity'],
    '--wandb-project', manifest['wandb_project'],
    '--wandb-run-name', manifest['wandb_run_name'],
    '--output-name', manifest['wandb_run_name'],
]
if REBUILD_CACHE:
    args.append('--rebuild-cache')
if COUNT_COVERAGE:
    args.append('--count-coverage')
if DRY_RUN:
    args.append('--dry-run')

write_command_script('train', train_py, args)
if RUN_TRAINING_IN_NOTEBOOK:
    run_script_streamed('train', train_py, args)
else:
    print('Training was not started in the notebook. Run the printed PowerShell command in a terminal.')
